In [1]:
# ============================================================
# STEP 2 - ATTRIBUTE MAPPING
# Standardized GPKG -> Mapped GPKG
# ============================================================

from pathlib import Path
import re
import shutil
from datetime import datetime

import pandas as pd
import geopandas as gpd

from tqdm.auto import tqdm

try:
    import pyogrio
except ImportError:
    pyogrio = None

try:
    import fiona
except ImportError:
    fiona = None

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

ROOT_DIR = Path.cwd()

# Output from STEP 1
INPUT_ROOT = ROOT_DIR / "Standardized_Output"

# Output STEP 2
OUTPUT_ROOT = ROOT_DIR / "Mapped_Output"

# Standard schema
STANDARD_SCHEMA = ROOT_DIR / "Standardized Seagrass Attribute Table.xlsx"

# ============================================================
# PROCESS MODE
# ============================================================

# "all"    -> all regions
# "region" -> one region only
MODE = "region"

REGION_NAME = "tambah"
# examples:
# Caribbean
# Global
# Indonesia
# Mediterranean
# Pacific
# etc.

# overwrite output
OVERWRITE = True

# Output separator when the target field has more than one source field
MULTI_FIELD_SEPARATOR = "<>"

# Input separators recognized in the Target field column
SOURCE_FIELD_SEPARATORS = ["<>", "&"]

# scan folder dataset sampai subfolder
SCAN_RECURSIVE = True

# geometry column
GEOMETRY_COLUMN = "geometry"

# Ignore case when searching for fields
IGNORE_CASE = True

# Fallback for manually shortened Excel sheet names
SHEET_FUZZY_MATCH_THRESHOLD = 0.92

# ============================================================
# Output Folder
# ============================================================

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Input :", INPUT_ROOT)
print("Output:", OUTPUT_ROOT)

Input : g:\Data Seagrass Global\Data Fix\Standardized_Output
Output: g:\Data Seagrass Global\Data Fix\Mapped_Output


In [3]:
# ============================================================
# LOAD STANDARD SCHEMA
# ============================================================

schema_df = pd.read_excel(STANDARD_SCHEMA)

schema_df.columns = schema_df.columns.str.strip()

COLUMN_NAME = "Column Name (snake_case)"

if COLUMN_NAME not in schema_df.columns:
    raise ValueError(
        f"Column '{COLUMN_NAME}' was not found in the schema."
    )

schema_df = schema_df.dropna(subset=[COLUMN_NAME]).copy()

schema_df[COLUMN_NAME] = (
    schema_df[COLUMN_NAME]
    .astype(str)
    .str.strip()
)

schema_df = schema_df[
    schema_df[COLUMN_NAME] != ""
]

schema_df = schema_df.drop_duplicates(
    subset=[COLUMN_NAME]
)

STANDARD_FIELDS = schema_df[COLUMN_NAME].tolist()

print("="*60)
print("Standard Fields")
print("="*60)
print(f"Field count : {len(STANDARD_FIELDS)}")
display(schema_df)

Standard Fields
Field count : 35


,Category,Column Name (snake_case),Data Type,Description
0,Metadata,data_id,String,Unique record identifier matching the composit...
1,Metadata,country_iso,String (Alpha-3),"ISO 3166-1 alpha-3 country code (e.g., IDN, MY..."
2,Metadata,quality_tier,String (Q1-Q4),Assigned data quality tier (Tier 1-4) determin...
3,Metadata,status,String,Complete/Incomplete (Complete = all entries ha...
4,Metadata,authors_name,String,"Primary data authority, citation authors"
5,Metadata,authors_institution,String,Institution of the data author
6,Spatial Attribute,longitude_x,Float,Geographic coordinate X in WGS84 Decimal Degre...
7,Spatial Attribute,latitude_y,Float,Geographic coordinate Y in WGS84 Decimal Degre...
8,Spatial Attribute,original_crs,String,Original Coordinate Reference System/Datum bef...
9,Spatial Attribute,depth_m,Float,Depth in meters (positive value). Critical for...


In [4]:
# ============================================================
# REGION SELECTION
# ============================================================

if MODE.lower() == "all":

    REGION_DIRS = sorted([
        p for p in INPUT_ROOT.iterdir()
        if p.is_dir()
    ])

elif MODE.lower() == "region":

    region = INPUT_ROOT / REGION_NAME

    if not region.exists():
        raise FileNotFoundError(region)

    REGION_DIRS = [region]

else:
    raise ValueError("MODE must be 'all' or 'region'.")

print("="*60)
print("Regions to process")
print("="*60)

for r in REGION_DIRS:
    print("-", r.name)

Regions to process
- tambah


In [5]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_layer_name(name, max_len=63):

    name = re.sub(
        r"[^0-9A-Za-z_]+",
        "_",
        str(name)
    ).strip("_")

    if len(name) == 0:
        name = "layer"

    if name[0].isdigit():
        name = "layer_" + name

    return name[:max_len]


def remove_output(path):

    if not path.exists():
        return

    path.unlink()


def get_geometry_column(gdf):

    return gdf.geometry.name


def normalize(text):

    text = str(text).strip()

    if IGNORE_CASE:
        text = text.lower()

    return text


def relative_folder(region_dir, folder):

    return folder.relative_to(region_dir).as_posix()


def temp_gpkg_path(output_path):

    return output_path.with_name(output_path.stem + "__tmp" + output_path.suffix)


def resolve_mapping_excel(folder, direct_excel_files):

    if len(direct_excel_files) > 0:
        return direct_excel_files

    parent_excel = folder.parent / f"{folder.name}.xlsx"

    if parent_excel.exists():
        return [parent_excel]

    return []


def possible_sheet_keys(dataset_stem):

    stem = str(dataset_stem).strip()

    candidates = [
        stem,
        stem[:31],
        stem.replace("_Posidonia_Zona_", "_Posi_"),
        f"{stem}.shp",
        f"{stem}.shp"[:31]
    ]

    return list(dict.fromkeys([
        normalize(c)
        for c in candidates
        if c != ""
    ]))


def find_matching_sheet(dataset_stem, sheet_lookup):

    for key in possible_sheet_keys(dataset_stem):

        if key in sheet_lookup:
            return sheet_lookup[key]

    from difflib import SequenceMatcher

    dataset_key = normalize(dataset_stem)

    best_sheet = None
    best_score = 0

    for sheet_key, sheet_name in sheet_lookup.items():

        score = SequenceMatcher(
            None,
            dataset_key,
            sheet_key
        ).ratio()

        if score > best_score:
            best_score = score
            best_sheet = sheet_name

    if best_score >= SHEET_FUZZY_MATCH_THRESHOLD:
        return best_sheet

    return None

In [6]:
# ============================================================
# SCAN DATASET FOLDERS
# ============================================================

dataset_folders = []

for region_dir in REGION_DIRS:

    print(f"\nScanning region : {region_dir.name}")

    if SCAN_RECURSIVE:

        folders = sorted([
            p for p in region_dir.rglob("*")
            if p.is_dir()
        ])

    else:

        folders = sorted([
            p for p in region_dir.iterdir()
            if p.is_dir()
        ])

    for folder in folders:

        direct_excel_files = sorted(folder.glob("*.xlsx"))
        excel_files = resolve_mapping_excel(folder, direct_excel_files)
        gpkg_files = sorted(folder.glob("*.gpkg"))

        if len(excel_files) == 0 and len(gpkg_files) == 0:
            continue

        dataset_folders.append({

            "region": region_dir.name,

            "folder_name": relative_folder(region_dir, folder),

            "folder_path": folder,

            "excel_files": excel_files,

            "excel_source": "direct" if len(direct_excel_files) > 0 else "parent_by_folder_name",

            "gpkg_files": gpkg_files

        })

print("="*60)
print("Folders found :", len(dataset_folders))


Scanning region : tambah
Folders found : 1


In [7]:
# ============================================================
# VALIDATE DATASET FOLDERS
# ============================================================

folder_log = []

valid_dataset_folders = []

for item in dataset_folders:

    status = "OK"
    message = ""

    if len(item["excel_files"]) == 0:

        status = "FAILED"
        message = "Excel mapping file was not found."

    elif len(item["excel_files"]) > 1:

        status = "FAILED"
        message = "More than one Excel file was found."

    elif len(item["gpkg_files"]) == 0:

        status = "FAILED"
        message = "No GPKG file was found."

    folder_log.append({

        "Region": item["region"],

        "Folder": item["folder_name"],

        "Excel": len(item["excel_files"]),

        "Excel_Path": str(item["excel_files"][0]) if len(item["excel_files"]) == 1 else "",

        "Excel_Source": item.get("excel_source", ""),

        "GPKG": len(item["gpkg_files"]),

        "Status": status,

        "Message": message

    })

    if status == "OK":
        valid_dataset_folders.append(item)

folder_log_df = pd.DataFrame(folder_log)

display(folder_log_df)

print("="*60)
print("Valid folders :", len(valid_dataset_folders))

,Region,Folder,Excel,Excel_Path,Excel_Source,GPKG,Status,Message
0,tambah,05_CCN_data,1,g:\Data Seagrass Global\Data Fix\Standardized_...,direct,1,OK,


Valid folders : 1


In [8]:
# ============================================================
# VALIDATE SHEETS
# ============================================================

dataset_manifest = []

sheet_log = []

for folder in valid_dataset_folders:

    excel_path = folder["excel_files"][0]

    workbook = pd.ExcelFile(excel_path)

    sheet_names = workbook.sheet_names

    sheet_lookup = {
        normalize(s): s
        for s in sheet_names
    }

    gpkg_sheet_keys = set()
    matched_sheet_keys = set()

    for gpkg in folder["gpkg_files"]:

        stem = gpkg.stem

        gpkg_sheet_keys.update(possible_sheet_keys(stem))

        matched_sheet = find_matching_sheet(stem, sheet_lookup)

        if matched_sheet is None:

            sheet_log.append({

                "Region": folder["region"],

                "Folder": folder["folder_name"],

                "Dataset": gpkg.name,

                "Sheet": "",

                "Status": "WARNING",

                "Message": "Sheet was not found"

            })

            continue

        matched_sheet_keys.add(normalize(matched_sheet))

        dataset_manifest.append({

            "region": folder["region"],

            "folder_name": folder["folder_name"],

            "folder_path": folder["folder_path"],

            "excel_path": excel_path,

            "sheet_name": matched_sheet,

            "gpkg_path": gpkg

        })

        sheet_log.append({

            "Region": folder["region"],

            "Folder": folder["folder_name"],

            "Dataset": gpkg.name,

            "Sheet": matched_sheet,

            "Status": "OK",

            "Message": ""

        })

    #
    # check sheets that do not have a dataset
    #

    for sheet in sheet_names:

        if normalize(sheet) not in gpkg_sheet_keys and normalize(sheet) not in matched_sheet_keys:

            sheet_log.append({

                "Region": folder["region"],

                "Folder": folder["folder_name"],

                "Dataset": "",

                "Sheet": sheet,

                "Status": "WARNING",

                "Message": "Sheet has no GPKG"

            })

sheet_log_df = pd.DataFrame(sheet_log)

display(sheet_log_df)

,Region,Folder,Dataset,Sheet,Status,Message
0,tambah,05_CCN_data,CCN_data.gpkg,CCN_data,OK,


In [9]:
# ============================================================
# DATASET MANIFEST
# ============================================================

manifest_df = pd.DataFrame(dataset_manifest)

print("="*60)
print("Datasets ready to process")
print("="*60)

display(manifest_df)

print()

print(f"Dataset count : {len(manifest_df)}")

Datasets ready to process


,region,folder_name,folder_path,excel_path,sheet_name,gpkg_path
0,tambah,05_CCN_data,g:\Data Seagrass Global\Data Fix\Standardized_...,g:\Data Seagrass Global\Data Fix\Standardized_...,CCN_data,g:\Data Seagrass Global\Data Fix\Standardized_...



Dataset count : 1


In [10]:
# ============================================================
# PREPARE OUTPUT PATH
# ============================================================

manifest_df["output_folder"] = manifest_df.apply(

    lambda r:

        OUTPUT_ROOT /
        r["region"] /
        Path(r["folder_name"]),

    axis=1

)

manifest_df["output_path"] = manifest_df.apply(

    lambda r:

        r["output_folder"] /
        r["gpkg_path"].name,

    axis=1

)

for p in manifest_df["output_folder"]:

    p.mkdir(
        parents=True,
        exist_ok=True
    )

display(

    manifest_df[
        [
            "region",
            "folder_name",
            "gpkg_path",
            "sheet_name",
            "output_path"
        ]
    ]

)

,region,folder_name,gpkg_path,sheet_name,output_path
0,tambah,05_CCN_data,g:\Data Seagrass Global\Data Fix\Standardized_...,CCN_data,g:\Data Seagrass Global\Data Fix\Mapped_Output...


In [11]:
# ============================================================
# LOAD MAPPING SHEET
# ============================================================

def load_mapping_sheet(excel_path, sheet_name):

    df = pd.read_excel(
        excel_path,
        sheet_name=sheet_name
    )

    df.columns = df.columns.str.strip()

    required = [
        "Column Name (snake_case)",
        "Target field",
        "Fill"
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{sheet_name}: required columns were not found : {missing}"
        )

    df = df[required].copy()

    df = df.fillna("")

    return df

In [12]:
# ============================================================
# FIND FIELD
# ============================================================

def field_key(field_name):

    key = str(field_name).strip()

    if IGNORE_CASE:
        key = key.lower()

    return key


def build_field_lookup(gdf):

    lookup = {}

    for c in gdf.columns:

        lookup[field_key(c)] = c

    return lookup


def field_exists(field_name, lookup):

    return field_key(field_name) in lookup


def get_real_field(field_name, lookup):

    return lookup[field_key(field_name)]

In [13]:
# ============================================================
# CONCAT MULTIPLE FIELDS
# ============================================================

def split_source_fields(field_string):

    field_string = str(field_string).strip()

    for separator in SOURCE_FIELD_SEPARATORS:

        if separator in field_string:

            fields = [
                f.strip()
                for f in field_string.split(separator)
                if f.strip() != ""
            ]

            if len(fields) > 1:
                return fields, separator

    return [field_string], None


def is_multi_source_field(field_string):

    fields, _ = split_source_fields(field_string)

    return len(fields) > 1


def concat_fields(
    gdf,
    field_string,
    separator=MULTI_FIELD_SEPARATOR
):

    fields, input_separator = split_source_fields(field_string)

    lookup = build_field_lookup(gdf)

    real_fields = []

    missing_fields = []

    for f in fields:

        if field_exists(f, lookup):

            real_fields.append(
                get_real_field(f, lookup)
            )

        else:

            missing_fields.append(f)

    #
    # if there are no valid fields at all
    #

    if len(real_fields) == 0:

        return None, missing_fields

    def combine(row):

        values = []

        for c in real_fields:

            value = row[c]

            if pd.isna(value):
                continue

            value = str(value).strip()

            if value == "":
                continue

            values.append(value)

        return f" {separator} ".join(values)

    return gdf.apply(combine, axis=1), missing_fields

In [14]:
# ============================================================
# COPY SINGLE FIELD
# ============================================================

def copy_field(gdf, source_field):

    lookup = build_field_lookup(gdf)

    if not field_exists(
        source_field,
        lookup
    ):

        return None

    real = get_real_field(
        source_field,
        lookup
    )

    return gdf[real]

In [15]:
# ============================================================
# CLEANUP FIELD
# ============================================================

def ensure_standard_fields(
    gdf,
    standard_fields
):

    created = []

    for field in standard_fields:

        if field not in gdf.columns:

            gdf[field] = pd.NA
            created.append(field)

    return gdf, created


def cleanup_fields(
    gdf,
    standard_fields
):

    geometry = gdf.geometry.name

    keep = set(standard_fields)

    keep.add(geometry)

    drop = [

        c

        for c in gdf.columns

        if c not in keep

    ]

    gdf = gdf.drop(
        columns=drop,
        errors="ignore"
    )

    return gdf

In [16]:
# ============================================================
# ATTRIBUTE MAPPING ENGINE
# ============================================================

def apply_mapping(
    gdf,
    mapping_df,
    standard_fields
):
    """
    Rules:
    1. Target field takes priority
    2. If Target is empty -> use Fill
    3. Multiple targets are separated using the separators defined in SOURCE_FIELD_SEPARATORS
    4. A missing source field is only a warning
    5. After processing, remove all old fields
    """

    logs = []

    gdf, created_fields = ensure_standard_fields(
        gdf,
        standard_fields
    )

    for field in created_fields:

        logs.append({

            "Field": field,
            "Status": "INFO",
            "Message": "Standard field created as NULL"

        })

    for _, row in mapping_df.iterrows():

        target = str(row["Column Name (snake_case)"]).strip()

        source = str(row["Target field"]).strip()
        fill = row["Fill"]

        # --------------------------------------------------
        # Clean empty values from Excel
        # --------------------------------------------------

        if source.lower() == "nan":
            source = ""

        if pd.isna(fill):
            fill = ""
        else:
            fill = str(fill).strip()
            if fill.lower() == "nan":
                fill = ""

        # ==================================================
        # PRIORITY 1
        # COPY FROM SOURCE FIELD
        # ==================================================

        if source != "":

            # ----------------------------------------------
            # MULTI FIELD
            # ----------------------------------------------

            if is_multi_source_field(source):

                values, missing = concat_fields(
                    gdf,
                    source,
                    separator=MULTI_FIELD_SEPARATOR
                )

                if values is None:

                    logs.append({

                        "Field": target,
                        "Status": "WARNING",
                        "Message": f"All source fields were not found: {', '.join(missing)}"

                    })

                    continue

                gdf[target] = values

                if len(missing) > 0:

                    logs.append({

                        "Field": target,
                        "Status": "WARNING",
                        "Message": f"Concat succeeded, but the following fields were not found: {', '.join(missing)}"

                    })

                else:

                    logs.append({

                        "Field": target,
                        "Status": "OK",
                        "Message": "Concat"

                    })

                continue

            # ----------------------------------------------
            # SINGLE FIELD
            # ----------------------------------------------

            values = copy_field(
                gdf,
                source
            )

            if values is None:

                logs.append({

                    "Field": target,
                    "Status": "WARNING",
                    "Message": f"Source field '{source}' was not found"

                })

                continue

            gdf[target] = values

            logs.append({

                "Field": target,
                "Status": "OK",
                "Message": f"Copied from '{source}'"

            })

            continue

        # ==================================================
        # PRIORITY 2
        # FILL CONSTANT
        # ==================================================

        if fill != "":

            gdf[target] = fill

            logs.append({

                "Field": target,
                "Status": "OK",
                "Message": f"Fill = {fill}"

            })

            continue

        # ==================================================
        # NO SOURCE AND NO FILL
        # ==================================================

        logs.append({

            "Field": target,
            "Status": "OK",
            "Message": "Remain NULL"

        })

    # ======================================================
    # Remove all old fields
    # ======================================================

    gdf = cleanup_fields(
        gdf,
        standard_fields
    )

    return gdf, pd.DataFrame(logs)

In [17]:
# ============================================================
# PROCESS DATASET
# ============================================================

processing_log = []

field_logs = []

for idx, row in tqdm(
    manifest_df.iterrows(),
    total=len(manifest_df),
    desc="Processing"
):

    dataset_name = row["gpkg_path"].name
    dataset_path = str(row["gpkg_path"].relative_to(INPUT_ROOT))

    print("="*70)
    print(dataset_name)
    print("="*70)

    try:

        output = row["output_path"]

        if output.exists() and not OVERWRITE:

            processing_log.append({

                "Dataset": dataset_name,
                "Dataset_Path": dataset_path,
                "Status": "SKIPPED",
                "Features": None,
                "Output": str(output),
                "Message": "Output already exists and OVERWRITE=False"

            })

            continue

        #
        # read gpkg
        #

        gdf = gpd.read_file(
            row["gpkg_path"]
        )

        #
        # mapping
        #

        mapping_df = load_mapping_sheet(

            row["excel_path"],

            row["sheet_name"]

        )

        #
        # apply
        #

        gdf, mapping_log = apply_mapping(

            gdf,

            mapping_df,

            STANDARD_FIELDS

        )

        #
        # save
        #

        tmp_output = temp_gpkg_path(output)

        if tmp_output.exists():

            tmp_output.unlink()

        gdf.to_file(

            tmp_output,

            driver="GPKG"

        )

        if output.exists():

            output.unlink()

        shutil.move(
            str(tmp_output),
            str(output)
        )

        #
        # add dataset information
        #

        mapping_log.insert(
            0,
            "Dataset",
            dataset_name
        )

        mapping_log.insert(
            1,
            "Dataset_Path",
            dataset_path
        )

        field_logs.append(
            mapping_log
        )

        processing_log.append({

            "Dataset": dataset_name,

            "Dataset_Path": dataset_path,

            "Status": "SUCCESS",

            "Features": len(gdf),

            "Output": str(output),

            "Message": ""

        })

    except Exception as e:

        processing_log.append({

            "Dataset": dataset_name,

            "Dataset_Path": dataset_path,

            "Status": "FAILED",

            "Features":None,

            "Output":"",

            "Message":str(e)

        })

        print(e)

Processing:   0%|          | 0/1 [00:00<?, ?it/s]

CCN_data.gpkg


In [18]:
# ============================================================
# BUILD LOG
# ============================================================

processing_log_df = pd.DataFrame(
    processing_log
)

if len(field_logs)>0:

    field_log_df = pd.concat(

        field_logs,

        ignore_index=True

    )

else:

    field_log_df = pd.DataFrame()

In [19]:
# ============================================================
# SUMMARY
# ============================================================

print()

print("="*60)

print("PROCESS SUMMARY")

print("="*60)

display(

    processing_log_df

)

print()

display(processing_log_df

    .groupby("Status")

    .size()

    .rename("Count")

)


PROCESS SUMMARY


,Dataset,Dataset_Path,Status,Features,Output,Message
0,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,SUCCESS,1060,g:\Data Seagrass Global\Data Fix\Mapped_Output...,


Status
SUCCESS    1
Name: Count, dtype: int64

In [20]:
# ============================================================
# SAVE LOG
# ============================================================

log_dir = OUTPUT_ROOT / "_logs"

log_dir.mkdir(

    parents=True,

    exist_ok=True

)

processing_log_df.to_csv(

    log_dir / "processing_log.csv",

    index=False,

    encoding="utf-8-sig"

)

field_log_df.to_csv(

    log_dir / "field_mapping_log.csv",

    index=False,

    encoding="utf-8-sig"

)

folder_log_df.to_csv(

    log_dir / "folder_validation_log.csv",

    index=False,

    encoding="utf-8-sig"

)

sheet_log_df.to_csv(

    log_dir / "sheet_validation_log.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("Logs saved :")

print(log_dir)


Logs saved :
g:\Data Seagrass Global\Data Fix\Mapped_Output\_logs


## Statistic

In [21]:
# ============================================================
# DATASET STATISTICS
# ============================================================

dataset_statistics = []

for _, row in processing_log_df.iterrows():

    if row["Status"] != "SUCCESS":
        continue

    dataset_statistics.append({

        "Dataset": row["Dataset"],

        "Features": row["Features"]

    })

dataset_statistics = pd.DataFrame(dataset_statistics)

display(dataset_statistics)

,Dataset,Features
0,CCN_data.gpkg,1060


In [22]:
# ============================================================
# FIELD SUMMARY
# ============================================================

print("="*60)
print("FIELD MAPPING SUMMARY")
print("="*60)

display(

    field_log_df

    .groupby(

        ["Status","Message"]

    )

    .size()

    .rename("Count")

    .reset_index()

)

FIELD MAPPING SUMMARY


,Status,Message,Count
0,OK,Copied from 'core_count',1
1,OK,Copied from 'core_habit',2
2,OK,Copied from 'core_latit',1
3,OK,Copied from 'core_longi',1
4,OK,Copied from 'core_year',1
5,OK,Copied from 'site_id',1
6,OK,Copied from 'species_sp',2
7,OK,Fill = COASTAL CARBON NETWORK (CCN),1
8,OK,Fill = Coastal Carbon Atlas,1
9,OK,Fill = Data not from a field survey. No field ...,1


In [23]:
# ============================================================
# QC REPORT
# ============================================================

qc_log = []

for _, row in manifest_df.iterrows():

    output = row["output_path"]
    dataset_path = str(row["gpkg_path"].relative_to(INPUT_ROOT))

    if not output.exists():
        continue

    gdf = gpd.read_file(output)

    for field in STANDARD_FIELDS:

        if field not in gdf.columns:

            qc_log.append({

                "Dataset": output.name,

                "Dataset_Path": dataset_path,

                "Field":field,

                "Status":"MISSING COLUMN"

            })

            continue

        values = gdf[field]

        all_empty = values.isna().all() or values.fillna("").astype(str).str.strip().eq("").all()

        if all_empty:

            qc_log.append({

                "Dataset": output.name,

                "Dataset_Path": dataset_path,

                "Field":field,

                "Status":"ALL NULL"

            })

qc_df = pd.DataFrame(qc_log)

display(qc_df)

,Dataset,Dataset_Path,Field,Status
0,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,depth_m,ALL NULL
1,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,original_survey_date,ALL NULL
2,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,survey_date,ALL NULL
3,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,data_origin,ALL NULL
4,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,data_analysis,ALL NULL
5,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,sampling_size_m2,ALL NULL
6,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,cover_seagrass_pct,ALL NULL
7,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,cover_coral_pct,ALL NULL
8,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,cover_algae_pct,ALL NULL
9,CCN_data.gpkg,tambah\05_CCN_data\CCN_data.gpkg,cover_substrate_pct,ALL NULL


In [24]:
qc_df.to_csv(

    log_dir/"qc_report.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("QC Report saved.")


QC Report saved.


In [25]:
print()

print("="*70)

print("ATTRIBUTE MAPPING FINISHED")

print("="*70)

print()

print("Dataset processed : ",len(manifest_df))

print("Success          : ",

      (processing_log_df["Status"]=="SUCCESS").sum())

print("Failed           : ",

      (processing_log_df["Status"]=="FAILED").sum())

print()

print("Output")

print(OUTPUT_ROOT)

print()

print("Logs")

print(log_dir)


ATTRIBUTE MAPPING FINISHED

Dataset processed :  1
Success          :  1
Failed           :  0

Output
g:\Data Seagrass Global\Data Fix\Mapped_Output

Logs
g:\Data Seagrass Global\Data Fix\Mapped_Output\_logs
